# 06 · Pandas drills — groupby, merge, reshape, dates

**Datasets:** `data/orders.csv` (5,000 orders) + `data/customers.csv` (400 customers).
**Covers:** the wrangling muscle memory — groupby, merge, pivot, window functions,
datetimes, ranking.
**Time yourself:** ~30 minutes, and try to keep each answer to one chained expression.

These are the "quick, get me X" questions that fill the first 10 minutes of a live
coding round. There is no modeling here — it's pure speed and fluency. The point is
to not have to think about `groupby` syntax while you're also thinking about the problem.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

orders = pd.read_csv('data/orders.csv', parse_dates=['order_date'])
customers = pd.read_csv('data/customers.csv', parse_dates=['signup_date'])

print('orders   ', orders.shape)
print('customers', customers.shape)
display(orders.head(3))
display(customers.head(3))

---

## Part A — Groupby

### Q1. Total revenue and number of orders per `category`, sorted by revenue descending.

<details><summary>hint</summary>

Named aggregation: `.agg(new_name=('col', 'func'))`.

</details>

In [ ]:
(orders.groupby('category')
       .agg(revenue=('revenue', 'sum'), n_orders=('order_id', 'count'))
       .sort_values('revenue', ascending=False)
       .round(2))

### Q2. Average order value per `channel` **and** `category` together. Return it as a wide
table with channels as rows and categories as columns.

<details><summary>hint</summary>

`pivot_table` — or `groupby([...]).mean().unstack()`, which is the same thing the long way round.

</details>

In [ ]:
orders.pivot_table(index='channel', columns='category', values='revenue',
                   aggfunc='mean').round(2)

### Q3. For each `category`, compute total revenue, mean revenue, median revenue, and the
number of *distinct customers* who bought from it. Four aggregations, one call.

<details><summary>hint</summary>

`'nunique'` is the aggregation you want for distinct counts.

</details>

In [ ]:
(orders.groupby('category')
       .agg(total=('revenue', 'sum'),
            mean=('revenue', 'mean'),
            median=('revenue', 'median'),
            n_customers=('customer_id', 'nunique'))
       .round(2))

### Q4. `revenue` has some NaNs. How many, and does `groupby().sum()` include or ignore them?
Prove it, then recompute `revenue` from `quantity * unit_price` where it's missing.

<details><summary>hint</summary>

The difference between `.count()` and `.size()` on a groupby is exactly the NaN handling. That's the whole answer.

</details>

In [ ]:
print('NaN revenue:', orders['revenue'].isna().sum())

# groupby().sum() skips NaN by default (skipna=True) -- it does NOT propagate.
print(orders.groupby('category')['revenue'].sum().round(2))
print(orders.groupby('category')['revenue'].count())   # count excludes NaN
print(orders.groupby('category')['revenue'].size())    # size includes NaN -- the difference

orders['revenue'] = orders['revenue'].fillna(orders['quantity'] * orders['unit_price'])
print('NaN after fill:', orders['revenue'].isna().sum())

# The count vs size gap is the trap: a silent skip means your "average revenue" is over a
# different denominator than you think.

---

## Part B — Merge

### Q5. Join `orders` to `customers` to get the country on every order. Use an inner join,
then check: did you lose rows? If so, why — and is that OK?

<details><summary>hint</summary>

Always compare `len()` before and after a join. `~df['k'].isin(other['k'])` finds the rows that fell out.

</details>

In [ ]:
merged = orders.merge(customers, on='customer_id', how='inner')
print('orders:', len(orders), '-> merged:', len(merged), '| lost:', len(orders) - len(merged))

orphans = orders[~orders['customer_id'].isin(customers['customer_id'])]
print('orphan orders:', len(orphans), 'from', orphans['customer_id'].nunique(),
      'unknown customer_ids:', sorted(orphans['customer_id'].unique())[:5], '...')

# ~3% of orders reference customer_ids (901-920) that don't exist in customers.csv --
# a broken foreign key. An inner join silently deletes them, and your revenue total
# quietly drops by 3% with no error.
# Whether it's OK depends: for "revenue per country" you must exclude them anyway (no
# country known), but you should REPORT the gap, not swallow it. Never let an inner join
# be the thing that decides your row count for you.

### Q6. Redo it as a left join and use the indicator to quantify the mismatch precisely.
Which join would you use in a revenue report, and why?

<details><summary>hint</summary>

`how='left', indicator=True` adds a `_merge` column with values `both` / `left_only`. It's the fastest join sanity check there is.

</details>

In [ ]:
check = orders.merge(customers, on='customer_id', how='left', indicator=True)
print(check['_merge'].value_counts())

lost_rev = check.loc[check['_merge'] == 'left_only', 'revenue'].sum()
print(f'revenue attached to unknown customers: ${lost_rev:,.2f} '
      f'({lost_rev / check["revenue"].sum():.1%} of total)')

# Left join + indicator for the *check*: it keeps every order and labels the unmatched
# ones, so you can measure the damage instead of guessing.
# For the report itself: still left join, with the unmatched bucketed as 'Unknown'
# country. Total revenue then reconciles with finance's number, and the gap is visible on
# the face of the report rather than hidden in a where-clause.

### Q7. Using the merged data: total revenue per `country`, and the average revenue per customer in each country.

<details><summary>hint</summary>

`.assign(new=lambda d: ...)` lets you derive a column from the aggregation you just built, without breaking the chain.

</details>

In [ ]:
(merged.groupby('country')
       .agg(total_revenue=('revenue', 'sum'),
            n_customers=('customer_id', 'nunique'),
            n_orders=('order_id', 'count'))
       .assign(rev_per_customer=lambda d: d['total_revenue'] / d['n_customers'])
       .sort_values('total_revenue', ascending=False)
       .round(2))

---

## Part C — Dates

### Q8. Monthly total revenue. Then find the single best month.

<details><summary>hint</summary>

`groupby(df['order_date'].dt.to_period('M'))`, or `.set_index('order_date').resample(...)` — but check your pandas version for the resample alias ('M' vs 'ME').

</details>

In [ ]:
monthly = (orders.groupby(orders['order_date'].dt.to_period('M'))['revenue']
                 .sum()
                 .round(2))
print(monthly.head(12))
print('\nbest month:', monthly.idxmax(), f'${monthly.max():,.2f}')

# Portability note worth knowing: `.set_index('order_date').resample('ME')` does the same
# thing, but the alias changed -- 'M' in pandas < 2.2, 'ME' from 2.2 on, and each raises
# on the other version. `dt.to_period('M')` is spelled the same on every version.

### Q9. Revenue by day of week. Is the weekend different? Show the day names, in calendar
order, not alphabetical.

<details><summary>hint</summary>

`.dt.day_name()` gives the string. `.reindex(list_of_days)` forces calendar order — otherwise you get Friday first.

</details>

In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow = (orders.assign(dow=orders['order_date'].dt.day_name())
             .groupby('dow')['revenue']
             .agg(['sum', 'mean', 'count'])
             .reindex(dow_order)
             .round(2))
display(dow)

# The dates were generated uniformly, so there is no real weekday effect here -- and
# saying "this looks flat, ~1/7 each, I don't believe there's a signal" is the right
# answer. Inventing a story about a Tuesday bump would be the wrong one.

### Q10. For each customer, how many days passed between their signup and their **first**
order? Show the distribution.

<details><summary>hint</summary>

Subtracting two datetime columns gives a timedelta; `.dt.days` turns it into an integer.

</details>

In [ ]:
first_order = merged.groupby('customer_id')['order_date'].min().rename('first_order')
gap = (customers.set_index('customer_id')
                .join(first_order)
                .assign(days_to_first=lambda d: (d['first_order'] - d['signup_date']).dt.days))

print(gap['days_to_first'].describe().round(1))

# Negative values are possible here because the two dates were generated independently --
# in real data a negative "days to first order" means an order predates the signup, which
# is a data-integrity bug worth flagging, not a number to take the mean of.

### Q11. Compute a 3-month rolling average of monthly revenue.

<details><summary>hint</summary>

`.rolling(3).mean()`. The first two rows are NaN by design — there's no window yet.

</details>

In [ ]:
monthly_df = monthly.to_frame('revenue')
monthly_df['rolling_3m'] = monthly_df['revenue'].rolling(3).mean().round(2)
monthly_df.head(8)

---

## Part D — Window functions & ranking

### Q12. What share of each category's revenue does each `channel` contribute? You need the
group total broadcast back onto every row — that's `transform`, not `agg`.

<details><summary>hint</summary>

`transform('sum')` returns a Series the same length as the input, so you can divide by it. `agg` collapses; `transform` broadcasts.

</details>

In [ ]:
grouped = orders.groupby(['category', 'channel'])['revenue'].sum().reset_index()
grouped['category_total'] = grouped.groupby('category')['revenue'].transform('sum')
grouped['pct_of_category'] = (grouped['revenue'] / grouped['category_total'] * 100).round(1)
grouped.sort_values(['category', 'pct_of_category'], ascending=[True, False])

### Q13. Find the **top 2 customers by total revenue within each country**. This is the classic
'top-N per group' question — they will ask it.

<details><summary>hint</summary>

Sort descending, then `.groupby(...).head(2)` — head on a groupby is per-group. Know the `.rank()` version too; it's what they mean if they say 'window function'.

</details>

In [ ]:
cust_rev = (merged.groupby(['country', 'customer_id'])['revenue']
                  .sum()
                  .reset_index())
top2 = (cust_rev.sort_values('revenue', ascending=False)
                .groupby('country')
                .head(2)
                .sort_values(['country', 'revenue'], ascending=[True, False]))
display(top2.round(2))

# Alternative with an explicit rank, which is closer to the SQL window-function answer:
cust_rev['rank'] = cust_rev.groupby('country')['revenue'].rank(method='dense',
                                                               ascending=False)
display(cust_rev[cust_rev['rank'] <= 2].sort_values(['country', 'rank']).round(2))

### Q14. Flag each customer's orders with a running cumulative revenue total, ordered by date.
Show it for one customer to prove it works.

<details><summary>hint</summary>

Sort by the group key and the date FIRST, then `groupby(...).cumsum()`. Without the sort your cumulative total is in arbitrary order and silently wrong.

</details>

In [ ]:
merged = merged.sort_values(['customer_id', 'order_date'])
merged['cumulative_revenue'] = merged.groupby('customer_id')['revenue'].cumsum()

example = merged['customer_id'].value_counts().index[0]
display(merged[merged['customer_id'] == example]
        [['customer_id', 'order_date', 'revenue', 'cumulative_revenue']].head(10).round(2))

### Q15. For each customer, compute the number of days since their **previous** order
(`NaN` for their first). Then: what's the median gap between orders?

<details><summary>hint</summary>

`.shift(1)` within a groupby is the lag. Same rule as cumsum: sort first.

</details>

In [ ]:
merged['prev_order'] = merged.groupby('customer_id')['order_date'].shift(1)
merged['days_since_prev'] = (merged['order_date'] - merged['prev_order']).dt.days

print(merged['days_since_prev'].describe().round(1))
display(merged[merged['customer_id'] == example]
        [['order_date', 'prev_order', 'days_since_prev']].head(6))

---

## Part E — Reshape

### Q16. Build a customer-level feature table (the kind you'd feed a churn model): one row per
customer with total revenue, order count, average order value, distinct categories,
first and last order date, and their country and segment.

<details><summary>hint</summary>

One `groupby().agg()` with named aggregations, then `.join()` the customer attributes on the index. This shape — one row per entity — is what every tabular model wants.

</details>

In [ ]:
features = (merged.groupby('customer_id')
                  .agg(total_revenue=('revenue', 'sum'),
                       n_orders=('order_id', 'count'),
                       avg_order_value=('revenue', 'mean'),
                       n_categories=('category', 'nunique'),
                       first_order=('order_date', 'min'),
                       last_order=('order_date', 'max'))
                  .join(customers.set_index('customer_id')[['country', 'segment', 'age']])
                  .assign(customer_lifetime_days=lambda d: (d['last_order'] - d['first_order']).dt.days))

print(features.shape)
display(features.head().round(2))

### Q17. Add one column per category holding that customer's spend in it (`spend_electronics`,
`spend_books`, …). This is long → wide.

<details><summary>hint</summary>

`pivot_table` with `fill_value=0` — a customer who never bought books should get 0, not NaN. `.add_prefix()` names the columns.

</details>

In [ ]:
cat_spend = (merged.pivot_table(index='customer_id', columns='category',
                                values='revenue', aggfunc='sum', fill_value=0)
                   .add_prefix('spend_'))
features = features.join(cat_spend)
display(features.head().round(2))

### Q18. Reverse it: take the `spend_*` columns back to long format with one row per
(customer, category) — dropping the zeros.

<details><summary>hint</summary>

`.melt(id_vars=..., value_vars=...)` is the inverse of pivot. `str.removeprefix` strips the prefix back off.

</details>

In [ ]:
long = (features.reset_index()
                .melt(id_vars='customer_id',
                      value_vars=[c for c in features.columns if c.startswith('spend_')],
                      var_name='category', value_name='spend')
                .query('spend > 0')
                .assign(category=lambda d: d['category'].str.removeprefix('spend_')))
print(long.shape)
display(long.head().round(2))

---

## Debrief — the five that come up every time

1. `groupby().agg(name=('col', 'fn'))` — named aggregation. Know it cold.
2. `transform` vs `agg`. Broadcast vs collapse. Any "% of group total" question is
   `transform`.
3. **Top-N per group**: `sort_values().groupby().head(n)`, and the `.rank()` variant.
4. **Check every merge.** Row count before, row count after, `indicator=True` if they
   differ. An inner join deleting 3% of your revenue in silence is a real outage.
5. `shift` / `cumsum` / `rolling` — always sort by the group and time key first.

**The sneaky one:** `.count()` vs `.size()` on a groupby differ by exactly the NaNs.